In [1]:
import pandas as pd
import numpy as np
from pyecharts import options as opts
from pyecharts.charts import (Bar, Line, Pie, Grid, Page, Table, Timeline,
                             Scatter, EffectScatter, Map, ThemeRiver)
from pyecharts.globals import ThemeType
import os
from pyecharts.commons.utils import JsCode

# 读取Excel数据
try:
    df = pd.read_excel('./中国电影网电影_data (1).xlsx')
    print(f"成功读取数据，共{len(df)}条记录")
    df.head()
except Exception as e:
    print(f"读取数据失败: {e}")
    # 创建模拟数据用于演示
    df = pd.DataFrame({
        '电影名': [f'电影{i}' for i in range(1, 2625)],
        '累计票房': np.random.uniform(1000, 130000, 2624),
        '导演': ['导演' + str(i % 100) for i in range(2624)],
        '主演': ['主演' + str(i % 200) for i in range(2624)],
        '上映时间': pd.date_range(start='2010-01-01', end='2022-12-31', periods=2624),
        '国家及地区': np.random.choice(['中国', '中国/中国香港', '中国/美国', '美国', '英国'], 2624),
        '发行公司': ['公司' + str(i % 50) for i in range(2624)],
        '类型': np.random.choice(['喜剧', '动作', '爱情', '科幻', '悬疑', '剧情', '动画'], 2624)
    })
    df['累计票房'] = df['累计票房'].round(1)
    df['上映时间'] = df['上映时间'].dt.strftime('%Y-%m-%d')

# 数据预处理
df['累计票房(万)'] = df['累计票房'].astype(float)
df['上映年份'] = pd.to_datetime(df['上映时间']).dt.year
df['国家'] = df['国家及地区'].str.split('/').str[0]
df['主要类型'] = df['类型'].str.split('/').str[0]

# 1. 业绩总览：累计票房收入（亿元）、各年度占比
total_box_office = df['累计票房(万)'].sum() / 10000  # 转换为亿元

# 计算各年度票房占比
yearly_box_office = df.groupby('上映年份')['累计票房(万)'].sum()
total_yearly = yearly_box_office.sum()
yearly_percentage = (yearly_box_office / total_yearly * 100).round(2)

# 2016-2018年度占比
years = [2016, 2017, 2018]
percentage_data = []
for year in years:
    if year in yearly_percentage.index:
        percentage_data.append((f"{year}年度", yearly_percentage[year]))
    else:
        percentage_data.append((f"{year}年度", 0))

# 2. 左一：票房前十影片排行榜
top10_box_office = df.nlargest(10, '累计票房(万)', keep='all')
top10_box_office = top10_box_office.sort_values('累计票房(万)', ascending=False)

# 3. 左二：各个国家影片数量的柱状图
country_movie_count = df['国家'].value_counts().reset_index()
country_movie_count.columns = ['国家', '影片数量']

# 4. 中二：各个年度影片票房总额和影片数量（柱状+线性图表）
yearly_stats = df.groupby('上映年份').agg(
    票房总额(万)=('累计票房(万)', 'sum'),
    影片数量=('电影名', 'count')
).reset_index()

# 5. 中三：各个电影公司的影片数量和票房总额
company_stats = df.groupby('发行公司').agg(
    影片数量=('电影名', 'count'),
    票房总额(万)=('累计票房(万)', 'sum')
).reset_index()
company_stats = company_stats.nlargest(10, '影片数量', keep='all')

# 6. 右一：纯国产电影各个年度的票房总额和数量走势
chinese_movies = df[df['国家'] == '中国']
chinese_yearly = chinese_movies.groupby('上映年份').agg(
    票房总额(万)=('累计票房(万)', 'sum'),
    影片数量=('电影名', 'count')
).reset_index()

# 7. 右二：各个年度各个影片类别的票房数据表
yearly_genre_box_office = df.groupby(['上映年份', '主要类型'])['累计票房(万)'].sum().reset_index()

# 8. 右三：各个类别的电影数量饼图
genre_movie_count = df['主要类型'].value_counts().reset_index()
genre_movie_count.columns = ['类型', '影片数量']

# 创建各个图表

# 1. 业绩总览图表
def create_overview_chart():
    pie = (
        Pie(init_opts=opts.InitOpts(width="100%", height="300px", theme=ThemeType.LIGHT))
        .add(
            "票房占比",
            percentage_data,
            radius=["30%", "75%"],
            label_opts=opts.LabelOpts(formatter="{b}: {d}%")
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(
                title="业绩总览",
                subtitle=f"累计票房收入: {total_box_office:.2f} 亿元",
                pos_left="center",
                title_textstyle_opts=opts.TextStyleOpts(font_size=20)
            ),
            legend_opts=opts.LegendOpts(pos_bottom="1%", pos_left="center"),
            toolbox_opts=opts.ToolboxOpts(is_show=False)
        )
    )
    return pie

# 2. 票房前十影片排行榜
def create_top10_chart():
    bar = (
        Bar(init_opts=opts.InitOpts(width="100%", height="400px", theme=ThemeType.LIGHT))
        .add_xaxis(top10_box_office['电影名'].tolist())
        .add_yaxis("累计票房(万)", top10_box_office['累计票房(万)'].tolist(), color='skyblue')
        .reversal_axis()
        .set_series_opts(
            label_opts=opts.LabelOpts(position="right")
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(title="票房前十影片排行榜"),
            xaxis_opts=opts.AxisOpts(name="累计票房(万)"),
            yaxis_opts=opts.AxisOpts(name="电影名"),
            toolbox_opts=opts.ToolboxOpts(is_show=False)
        )
    )
    return bar

# 3. 各个国家影片数量柱状图
def create_country_bar_chart():
    bar = (
        Bar(init_opts=opts.InitOpts(width="100%", height="400px", theme=ThemeType.LIGHT))
        .add_xaxis(country_movie_count['国家'].tolist())
        .add_yaxis("影片数量", country_movie_count['影片数量'].tolist(), color='orange')
        .set_global_opts(
            title_opts=opts.TitleOpts(title="各个国家影片数量对比"),
            xaxis_opts=opts.AxisOpts(name="国家"),
            yaxis_opts=opts.AxisOpts(name="影片数量"),
            toolbox_opts=opts.ToolboxOpts(is_show=False)
        )
    )
    return bar

# 4. 各年度票房总额和数量图表
def create_yearly_combined_chart():
    bar = (
        Bar(init_opts=opts.InitOpts(width="100%", height="400px", theme=ThemeType.LIGHT))
        .add_xaxis(yearly_stats['上映年份'].tolist())
        .add_yaxis("票房总额(万)", yearly_stats['票房总额(万)'].tolist(), color='green')
        .set_global_opts(
            title_opts=opts.TitleOpts(title="各年度影片票房总额和数量"),
            xaxis_opts=opts.AxisOpts(name="上映年份"),
            yaxis_opts=opts.AxisOpts(name="票房总额(万)"),
            toolbox_opts=opts.ToolboxOpts(is_show=False)
        )
    )
    
    line = (
        Line()
        .add_xaxis(yearly_stats['上映年份'].tolist())
        .add_yaxis("影片数量", yearly_stats['影片数量'].tolist(), color='red',
                  yaxis_index=1)
        .set_global_opts(
            yaxis_opts=opts.AxisOpts(
                name="影片数量",
                type_="value",
                min_=0,
                max_=yearly_stats['影片数量'].max() * 1.1,
                position="right",
                axisline_opts=opts.AxisLineOpts(
                    linestyle_opts=opts.LineStyleOpts(color="red")
                ),
                axistick_opts=opts.AxisTickOpts(is_show=True),
                axislabel_opts=opts.LabelOpts(formatter="{value} 部")
            )
        )
    )
    
    bar.overlap(line)
    return bar

# 5. 电影公司影片数量和票房总额
def create_company_chart():
    line1 = (
        Line(init_opts=opts.InitOpts(width="100%", height="400px", theme=ThemeType.LIGHT))
        .add_xaxis(company_stats['发行公司'].tolist())
        .add_yaxis("影片数量", company_stats['影片数量'].tolist(), color='blue')
        .set_global_opts(
            title_opts=opts.TitleOpts(title="电影公司影片数量和票房总额"),
            xaxis_opts=opts.AxisOpts(name="发行公司", axislabel_opts=opts.LabelOpts(rotate=45)),
            yaxis_opts=opts.AxisOpts(name="影片数量"),
            toolbox_opts=opts.ToolboxOpts(is_show=False)
        )
    )
    
    line2 = (
        Line()
        .add_xaxis(company_stats['发行公司'].tolist())
        .add_yaxis("票房总额(万)", company_stats['票房总额(万)'].tolist(), color='purple', yaxis_index=1)
        .set_global_opts(
            yaxis_opts=opts.AxisOpts(
                name="票房总额(万)",
                type_="value",
                min_=0,
                max_=company_stats['票房总额(万)'].max() * 1.1,
                position="right",
                axisline_opts=opts.AxisLineOpts(
                    linestyle_opts=opts.LineStyleOpts(color="purple")
                ),
                axistick_opts=opts.AxisTickOpts(is_show=True),
                axislabel_opts=opts.LabelOpts(formatter="{value} 万")
            )
        )
    )
    
    line1.overlap(line2)
    return line1

# 6. 纯国产电影年度走势
def create_chinese_movie_chart():
    line = (
        Line(init_opts=opts.InitOpts(width="100%", height="400px", theme=ThemeType.LIGHT))
        .add_xaxis(chinese_yearly['上映年份'].tolist())
        .add_yaxis("票房总额(万)", chinese_yearly['票房总额(万)'].tolist(), color='green')
        .add_yaxis("影片数量", chinese_yearly['影片数量'].tolist(), color='red', yaxis_index=1)
        .set_global_opts(
            title_opts=opts.TitleOpts(title="纯国产电影年度票房和数量走势"),
            xaxis_opts=opts.AxisOpts(name="上映年份"),
            yaxis_opts=opts.AxisOpts(name="票房总额(万)"),
            toolbox_opts=opts.ToolboxOpts(is_show=False)
        )
    )
    
    line.extend_axis(
        yaxis=opts.AxisOpts(
            name="影片数量",
            type_="value",
            min_=0,
            max_=chinese_yearly['影片数量'].max() * 1.1,
            position="right",
            axisline_opts=opts.AxisLineOpts(
                linestyle_opts=opts.LineStyleOpts(color="red")
            ),
            axistick_opts=opts.AxisTickOpts(is_show=True),
            axislabel_opts=opts.LabelOpts(formatter="{value} 部")
        )
    )
    return line

# 7. 年度类别票房数据表
def create_genre_table_chart():
    # 准备表格数据
    table_data = [["上映年份", "类型", "票房总额(万)"]]
    for _, row in yearly_genre_box_office.iterrows():
        table_data.append([row['上映年份'], row['主要类型'], f"{row['累计票房(万)']:.1f}"])
    
    table = (
        Table(init_opts=opts.InitOpts(width="100%", height="400px", theme=ThemeType.LIGHT))
        .add("年度类别票房", table_data)
        .set_global_opts(
            title_opts=opts.TitleOpts(title="各个年度各个类别的票房数据"),
            toolbox_opts=opts.ToolboxOpts(is_show=False)
        )
    )
    return table

# 8. 电影类型数量饼图
def create_genre_pie_chart():
    pie = (
        Pie(init_opts=opts.InitOpts(width="100%", height="400px", theme=ThemeType.LIGHT))
        .add(
            "类型分布",
            [list(z) for z in zip(genre_movie_count['类型'], genre_movie_count['影片数量'])],
            radius=["30%", "75%"],
            label_opts=opts.LabelOpts(formatter="{b}: {d}%")
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(title="各个类型电影数量占比"),
            legend_opts=opts.LegendOpts(pos_bottom="1%", pos_left="center"),
            toolbox_opts=opts.ToolboxOpts(is_show=False)
        )
    )
    return pie

# 创建大屏布局
def create_dashboard():
    # 初始化Page
    page = Page(layout=Page.DraggablePageLayout, page_title="中国电影数据可视化大屏")
    
    # 添加各个图表到Page
    page.add(
        create_overview_chart(),
        create_top10_chart(),
        create_country_bar_chart(),
        create_yearly_combined_chart(),
        create_company_chart(),
        create_chinese_movie_chart(),
        create_genre_table_chart(),
        create_genre_pie_chart()
    )
    
    # 保存为HTML文件
    page.render("./movie_data_dashboard.html")
    print("数据可视化大屏已保存为movie_data_dashboard.html")

# 执行函数创建大屏
create_dashboard()

SyntaxError: keyword can't be an expression (<ipython-input-1-79b1103b62bb>, line 64)

In [2]:
import pandas as pd
import numpy as np
from pyecharts import options as opts
from pyecharts.charts import (Bar, Line, Pie, Grid, Page, Table, Timeline,
                             Scatter, EffectScatter, Map, ThemeRiver)
from pyecharts.globals import ThemeType
import os
from pyecharts.commons.utils import JsCode

# 读取Excel数据
try:
    df = pd.read_excel('./中国电影网电影_data (1).xlsx')
    print(f"成功读取数据，共{len(df)}条记录")
    df.head()
except Exception as e:
    print(f"读取数据失败: {e}")
    # 创建模拟数据用于演示
    df = pd.DataFrame({
        '电影名': [f'电影{i}' for i in range(1, 2625)],
        '累计票房': np.random.uniform(1000, 130000, 2624),
        '导演': ['导演' + str(i % 100) for i in range(2624)],
        '主演': ['主演' + str(i % 200) for i in range(2624)],
        '上映时间': pd.date_range(start='2010-01-01', end='2022-12-31', periods=2624),
        '国家及地区': np.random.choice(['中国', '中国/中国香港', '中国/美国', '美国', '英国'], 2624),
        '发行公司': ['公司' + str(i % 50) for i in range(2624)],
        '类型': np.random.choice(['喜剧', '动作', '爱情', '科幻', '悬疑', '剧情', '动画'], 2624)
    })
    df['累计票房'] = df['累计票房'].round(1)
    df['上映时间'] = df['上映时间'].dt.strftime('%Y-%m-%d')

# 数据预处理
df['累计票房万'] = df['累计票房'].astype(float)
df['上映年份'] = pd.to_datetime(df['上映时间']).dt.year
df['国家'] = df['国家及地区'].str.split('/').str[0]
df['主要类型'] = df['类型'].str.split('/').str[0]

# 1. 业绩总览：累计票房收入（亿元）、各年度占比
total_box_office = df['累计票房万'].sum() / 10000  # 转换为亿元

# 计算各年度票房占比
yearly_box_office = df.groupby('上映年份')['累计票房万'].sum()
total_yearly = yearly_box_office.sum()
yearly_percentage = (yearly_box_office / total_yearly * 100).round(2)

# 2016-2018年度占比
years = [2016, 2017, 2018]
percentage_data = []
for year in years:
    if year in yearly_percentage.index:
        percentage_data.append((f"{year}年度", yearly_percentage[year]))
    else:
        percentage_data.append((f"{year}年度", 0))

# 2. 左一：票房前十影片排行榜
top10_box_office = df.nlargest(10, '累计票房万', keep='all')
top10_box_office = top10_box_office.sort_values('累计票房万', ascending=False)

# 3. 左二：各个国家影片数量的柱状图
country_movie_count = df['国家'].value_counts().reset_index()
country_movie_count.columns = ['国家', '影片数量']

# 4. 中二：各个年度影片票房总额和影片数量（柱状+线性图表）
yearly_stats = df.groupby('上映年份').agg(
    票房总额万=('累计票房万', 'sum'),
    影片数量=('电影名', 'count')
).reset_index()

# 5. 中三：各个电影公司的影片数量和票房总额
company_stats = df.groupby('发行公司').agg(
    影片数量=('电影名', 'count'),
    票房总额万=('累计票房万', 'sum')
).reset_index()
company_stats = company_stats.nlargest(10, '影片数量', keep='all')

# 6. 右一：纯国产电影各个年度的票房总额和数量走势
chinese_movies = df[df['国家'] == '中国']
chinese_yearly = chinese_movies.groupby('上映年份').agg(
    票房总额万=('累计票房万', 'sum'),
    影片数量=('电影名', 'count')
).reset_index()

# 7. 右二：各个年度各个影片类别的票房数据表
yearly_genre_box_office = df.groupby(['上映年份', '主要类型'])['累计票房万'].sum().reset_index()

# 8. 右三：各个类别的电影数量饼图
genre_movie_count = df['主要类型'].value_counts().reset_index()
genre_movie_count.columns = ['类型', '影片数量']

# 创建各个图表

# 1. 业绩总览图表
def create_overview_chart():
    pie = (
        Pie(init_opts=opts.InitOpts(width="100%", height="300px", theme=ThemeType.LIGHT))
        .add(
            "票房占比",
            percentage_data,
            radius=["30%", "75%"],
            label_opts=opts.LabelOpts(formatter="{b}: {d}%")
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(
                title="业绩总览",
                subtitle=f"累计票房收入: {total_box_office:.2f} 亿元",
                pos_left="center",
                title_textstyle_opts=opts.TextStyleOpts(font_size=20)
            ),
            legend_opts=opts.LegendOpts(pos_bottom="1%", pos_left="center"),
            toolbox_opts=opts.ToolboxOpts(is_show=False)
        )
    )
    return pie

# 2. 票房前十影片排行榜
def create_top10_chart():
    bar = (
        Bar(init_opts=opts.InitOpts(width="100%", height="400px", theme=ThemeType.LIGHT))
        .add_xaxis(top10_box_office['电影名'].tolist())
        .add_yaxis("累计票房万", top10_box_office['累计票房万'].tolist(), color='skyblue')
        .reversal_axis()
        .set_series_opts(
            label_opts=opts.LabelOpts(position="right")
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(title="票房前十影片排行榜"),
            xaxis_opts=opts.AxisOpts(name="累计票房万"),
            yaxis_opts=opts.AxisOpts(name="电影名"),
            toolbox_opts=opts.ToolboxOpts(is_show=False)
        )
    )
    return bar

# 3. 各个国家影片数量柱状图
def create_country_bar_chart():
    bar = (
        Bar(init_opts=opts.InitOpts(width="100%", height="400px", theme=ThemeType.LIGHT))
        .add_xaxis(country_movie_count['国家'].tolist())
        .add_yaxis("影片数量", country_movie_count['影片数量'].tolist(), color='orange')
        .set_global_opts(
            title_opts=opts.TitleOpts(title="各个国家影片数量对比"),
            xaxis_opts=opts.AxisOpts(name="国家"),
            yaxis_opts=opts.AxisOpts(name="影片数量"),
            toolbox_opts=opts.ToolboxOpts(is_show=False)
        )
    )
    return bar

# 4. 各年度票房总额和数量图表
def create_yearly_combined_chart():
    bar = (
        Bar(init_opts=opts.InitOpts(width="100%", height="400px", theme=ThemeType.LIGHT))
        .add_xaxis(yearly_stats['上映年份'].tolist())
        .add_yaxis("票房总额万", yearly_stats['票房总额万'].tolist(), color='green')
        .set_global_opts(
            title_opts=opts.TitleOpts(title="各年度影片票房总额和数量"),
            xaxis_opts=opts.AxisOpts(name="上映年份"),
            yaxis_opts=opts.AxisOpts(name="票房总额万"),
            toolbox_opts=opts.ToolboxOpts(is_show=False)
        )
    )
    
    line = (
        Line()
        .add_xaxis(yearly_stats['上映年份'].tolist())
        .add_yaxis("影片数量", yearly_stats['影片数量'].tolist(), color='red',
                  yaxis_index=1)
        .set_global_opts(
            yaxis_opts=opts.AxisOpts(
                name="影片数量",
                type_="value",
                min_=0,
                max_=yearly_stats['影片数量'].max() * 1.1,
                position="right",
                axisline_opts=opts.AxisLineOpts(
                    linestyle_opts=opts.LineStyleOpts(color="red")
                ),
                axistick_opts=opts.AxisTickOpts(is_show=True),
                axislabel_opts=opts.LabelOpts(formatter="{value} 部")
            )
        )
    )
    
    bar.overlap(line)
    return bar

# 5. 电影公司影片数量和票房总额
def create_company_chart():
    line1 = (
        Line(init_opts=opts.InitOpts(width="100%", height="400px", theme=ThemeType.LIGHT))
        .add_xaxis(company_stats['发行公司'].tolist())
        .add_yaxis("影片数量", company_stats['影片数量'].tolist(), color='blue')
        .set_global_opts(
            title_opts=opts.TitleOpts(title="电影公司影片数量和票房总额"),
            xaxis_opts=opts.AxisOpts(name="发行公司", axislabel_opts=opts.LabelOpts(rotate=45)),
            yaxis_opts=opts.AxisOpts(name="影片数量"),
            toolbox_opts=opts.ToolboxOpts(is_show=False)
        )
    )
    
    line2 = (
        Line()
        .add_xaxis(company_stats['发行公司'].tolist())
        .add_yaxis("票房总额万", company_stats['票房总额万'].tolist(), color='purple', yaxis_index=1)
        .set_global_opts(
            yaxis_opts=opts.AxisOpts(
                name="票房总额万",
                type_="value",
                min_=0,
                max_=company_stats['票房总额万'].max() * 1.1,
                position="right",
                axisline_opts=opts.AxisLineOpts(
                    linestyle_opts=opts.LineStyleOpts(color="purple")
                ),
                axistick_opts=opts.AxisTickOpts(is_show=True),
                axislabel_opts=opts.LabelOpts(formatter="{value} 万")
            )
        )
    )
    
    line1.overlap(line2)
    return line1

# 6. 纯国产电影年度走势
def create_chinese_movie_chart():
    line = (
        Line(init_opts=opts.InitOpts(width="100%", height="400px", theme=ThemeType.LIGHT))
        .add_xaxis(chinese_yearly['上映年份'].tolist())
        .add_yaxis("票房总额万", chinese_yearly['票房总额万'].tolist(), color='green')
        .add_yaxis("影片数量", chinese_yearly['影片数量'].tolist(), color='red', yaxis_index=1)
        .set_global_opts(
            title_opts=opts.TitleOpts(title="纯国产电影年度票房和数量走势"),
            xaxis_opts=opts.AxisOpts(name="上映年份"),
            yaxis_opts=opts.AxisOpts(name="票房总额万"),
            toolbox_opts=opts.ToolboxOpts(is_show=False)
        )
    )
    
    line.extend_axis(
        yaxis=opts.AxisOpts(
            name="影片数量",
            type_="value",
            min_=0,
            max_=chinese_yearly['影片数量'].max() * 1.1,
            position="right",
            axisline_opts=opts.AxisLineOpts(
                linestyle_opts=opts.LineStyleOpts(color="red")
            ),
            axistick_opts=opts.AxisTickOpts(is_show=True),
            axislabel_opts=opts.LabelOpts(formatter="{value} 部")
        )
    )
    return line

# 7. 年度类别票房数据表
def create_genre_table_chart():
    # 准备表格数据
    table_data = [["上映年份", "类型", "票房总额万"]]
    for _, row in yearly_genre_box_office.iterrows():
        table_data.append([row['上映年份'], row['主要类型'], f"{row['累计票房万']:.1f}"])
    
    table = (
        Table(init_opts=opts.InitOpts(width="100%", height="400px", theme=ThemeType.LIGHT))
        .add("年度类别票房", table_data)
        .set_global_opts(
            title_opts=opts.TitleOpts(title="各个年度各个类别的票房数据"),
            toolbox_opts=opts.ToolboxOpts(is_show=False)
        )
    )
    return table

# 8. 电影类型数量饼图
def create_genre_pie_chart():
    pie = (
        Pie(init_opts=opts.InitOpts(width="100%", height="400px", theme=ThemeType.LIGHT))
        .add(
            "类型分布",
            [list(z) for z in zip(genre_movie_count['类型'], genre_movie_count['影片数量'])],
            radius=["30%", "75%"],
            label_opts=opts.LabelOpts(formatter="{b}: {d}%")
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(title="各个类型电影数量占比"),
            legend_opts=opts.LegendOpts(pos_bottom="1%", pos_left="center"),
            toolbox_opts=opts.ToolboxOpts(is_show=False)
        )
    )
    return pie

# 创建大屏布局
def create_dashboard():
    # 初始化Page
    page = Page(layout=Page.DraggablePageLayout, page_title="中国电影数据可视化大屏")
    
    # 添加各个图表到Page
    page.add(
        create_overview_chart(),
        create_top10_chart(),
        create_country_bar_chart(),
        create_yearly_combined_chart(),
        create_company_chart(),
        create_chinese_movie_chart(),
        create_genre_table_chart(),
        create_genre_pie_chart()
    )
    
    # 保存为HTML文件
    page.render("./movie_data_dashboard.html")
    print("数据可视化大屏已保存为movie_data_dashboard.html")

# 执行函数创建大屏
create_dashboard()

ImportError: cannot import name 'Table' from 'pyecharts.charts' (C:\ProgramData\Anaconda3\lib\site-packages\pyecharts\charts\__init__.py)

In [3]:
import pandas as pd
import numpy as np
from pyecharts import options as opts
from pyecharts.charts import Bar, Line, Pie, Grid, Page, Timeline, Scatter, EffectScatter, Map
from pyecharts.globals import ThemeType
from pyecharts.components import Table  # 修正Table导入方式
import os

# 读取Excel数据
try:
    df = pd.read_excel('./中国电影网电影_data (1).xlsx')
    print(f"成功读取数据，共{len(df)}条记录")
    df.head()
except Exception as e:
    print(f"读取数据失败: {e}")
    # 创建模拟数据用于演示
    df = pd.DataFrame({
        '电影名': [f'电影{i}' for i in range(1, 2625)],
        '累计票房': np.random.uniform(1000, 130000, 2624),
        '导演': ['导演' + str(i % 100) for i in range(2624)],
        '主演': ['主演' + str(i % 200) for i in range(2624)],
        '上映时间': pd.date_range(start='2010-01-01', end='2022-12-31', periods=2624),
        '国家及地区': np.random.choice(['中国', '中国/中国香港', '中国/美国', '美国', '英国'], 2624),
        '发行公司': ['公司' + str(i % 50) for i in range(2624)],
        '类型': np.random.choice(['喜剧', '动作', '爱情', '科幻', '悬疑', '剧情', '动画'], 2624)
    })
    df['累计票房'] = df['累计票房'].round(1)
    df['上映时间'] = df['上映时间'].dt.strftime('%Y-%m-%d')

# 数据预处理
df['累计票房万'] = df['累计票房'].astype(float)
df['上映年份'] = pd.to_datetime(df['上映时间']).dt.year
df['国家'] = df['国家及地区'].str.split('/').str[0]
df['主要类型'] = df['类型'].str.split('/').str[0]

# 1. 业绩总览：累计票房收入（亿元）、各年度占比
total_box_office = df['累计票房万'].sum() / 10000  # 转换为亿元

# 计算各年度票房占比
yearly_box_office = df.groupby('上映年份')['累计票房万'].sum()
total_yearly = yearly_box_office.sum()
yearly_percentage = (yearly_box_office / total_yearly * 100).round(2)

# 2016-2018年度占比
years = [2016, 2017, 2018]
percentage_data = []
for year in years:
    if year in yearly_percentage.index:
        percentage_data.append((f"{year}年度", yearly_percentage[year]))
    else:
        percentage_data.append((f"{year}年度", 0))

# 2. 左一：票房前十影片排行榜
top10_box_office = df.nlargest(10, '累计票房万', keep='all')
top10_box_office = top10_box_office.sort_values('累计票房万', ascending=False)

# 3. 左二：各个国家影片数量的柱状图
country_movie_count = df['国家'].value_counts().reset_index()
country_movie_count.columns = ['国家', '影片数量']

# 4. 中二：各个年度影片票房总额和影片数量（柱状+线性图表）
yearly_stats = df.groupby('上映年份').agg(
    票房总额万=('累计票房万', 'sum'),
    影片数量=('电影名', 'count')
).reset_index()

# 5. 中三：各个电影公司的影片数量和票房总额
company_stats = df.groupby('发行公司').agg(
    影片数量=('电影名', 'count'),
    票房总额万=('累计票房万', 'sum')
).reset_index()
company_stats = company_stats.nlargest(10, '影片数量', keep='all')

# 6. 右一：纯国产电影各个年度的票房总额和数量走势
chinese_movies = df[df['国家'] == '中国']
chinese_yearly = chinese_movies.groupby('上映年份').agg(
    票房总额万=('累计票房万', 'sum'),
    影片数量=('电影名', 'count')
).reset_index()

# 7. 右二：各个年度各个影片类别的票房数据表
yearly_genre_box_office = df.groupby(['上映年份', '主要类型'])['累计票房万'].sum().reset_index()

# 8. 右三：各个类别的电影数量饼图
genre_movie_count = df['主要类型'].value_counts().reset_index()
genre_movie_count.columns = ['类型', '影片数量']

# 创建各个图表

# 1. 业绩总览图表
def create_overview_chart():
    pie = (
        Pie(init_opts=opts.InitOpts(width="100%", height="300px", theme=ThemeType.LIGHT))
        .add(
            "票房占比",
            percentage_data,
            radius=["30%", "75%"],
            label_opts=opts.LabelOpts(formatter="{b}: {d}%")
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(
                title="业绩总览",
                subtitle=f"累计票房收入: {total_box_office:.2f} 亿元",
                pos_left="center",
                title_textstyle_opts=opts.TextStyleOpts(font_size=20)
            ),
            legend_opts=opts.LegendOpts(pos_bottom="1%", pos_left="center"),
            toolbox_opts=opts.ToolboxOpts(is_show=False)
        )
    )
    return pie

# 2. 票房前十影片排行榜
def create_top10_chart():
    bar = (
        Bar(init_opts=opts.InitOpts(width="100%", height="400px", theme=ThemeType.LIGHT))
        .add_xaxis(top10_box_office['电影名'].tolist())
        .add_yaxis("累计票房万", top10_box_office['累计票房万'].tolist(), color='skyblue')
        .reversal_axis()
        .set_series_opts(
            label_opts=opts.LabelOpts(position="right")
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(title="票房前十影片排行榜"),
            xaxis_opts=opts.AxisOpts(name="累计票房万"),
            yaxis_opts=opts.AxisOpts(name="电影名"),
            toolbox_opts=opts.ToolboxOpts(is_show=False)
        )
    )
    return bar

# 3. 各个国家影片数量柱状图
def create_country_bar_chart():
    bar = (
        Bar(init_opts=opts.InitOpts(width="100%", height="400px", theme=ThemeType.LIGHT))
        .add_xaxis(country_movie_count['国家'].tolist())
        .add_yaxis("影片数量", country_movie_count['影片数量'].tolist(), color='orange')
        .set_global_opts(
            title_opts=opts.TitleOpts(title="各个国家影片数量对比"),
            xaxis_opts=opts.AxisOpts(name="国家"),
            yaxis_opts=opts.AxisOpts(name="影片数量"),
            toolbox_opts=opts.ToolboxOpts(is_show=False)
        )
    )
    return bar

# 4. 各年度票房总额和数量图表
def create_yearly_combined_chart():
    bar = (
        Bar(init_opts=opts.InitOpts(width="100%", height="400px", theme=ThemeType.LIGHT))
        .add_xaxis(yearly_stats['上映年份'].tolist())
        .add_yaxis("票房总额万", yearly_stats['票房总额万'].tolist(), color='green')
        .set_global_opts(
            title_opts=opts.TitleOpts(title="各年度影片票房总额和数量"),
            xaxis_opts=opts.AxisOpts(name="上映年份"),
            yaxis_opts=opts.AxisOpts(name="票房总额万"),
            toolbox_opts=opts.ToolboxOpts(is_show=False)
        )
    )
    
    line = (
        Line()
        .add_xaxis(yearly_stats['上映年份'].tolist())
        .add_yaxis("影片数量", yearly_stats['影片数量'].tolist(), color='red',
                  yaxis_index=1)
        .set_global_opts(
            yaxis_opts=opts.AxisOpts(
                name="影片数量",
                type_="value",
                min_=0,
                max_=yearly_stats['影片数量'].max() * 1.1,
                position="right",
                axisline_opts=opts.AxisLineOpts(
                    linestyle_opts=opts.LineStyleOpts(color="red")
                ),
                axistick_opts=opts.AxisTickOpts(is_show=True),
                axislabel_opts=opts.LabelOpts(formatter="{value} 部")
            )
        )
    )
    
    bar.overlap(line)
    return bar

# 5. 电影公司影片数量和票房总额
def create_company_chart():
    line1 = (
        Line(init_opts=opts.InitOpts(width="100%", height="400px", theme=ThemeType.LIGHT))
        .add_xaxis(company_stats['发行公司'].tolist())
        .add_yaxis("影片数量", company_stats['影片数量'].tolist(), color='blue')
        .set_global_opts(
            title_opts=opts.TitleOpts(title="电影公司影片数量和票房总额"),
            xaxis_opts=opts.AxisOpts(name="发行公司", axislabel_opts=opts.LabelOpts(rotate=45)),
            yaxis_opts=opts.AxisOpts(name="影片数量"),
            toolbox_opts=opts.ToolboxOpts(is_show=False)
        )
    )
    
    line2 = (
        Line()
        .add_xaxis(company_stats['发行公司'].tolist())
        .add_yaxis("票房总额万", company_stats['票房总额万'].tolist(), color='purple', yaxis_index=1)
        .set_global_opts(
            yaxis_opts=opts.AxisOpts(
                name="票房总额万",
                type_="value",
                min_=0,
                max_=company_stats['票房总额万'].max() * 1.1,
                position="right",
                axisline_opts=opts.AxisLineOpts(
                    linestyle_opts=opts.LineStyleOpts(color="purple")
                ),
                axistick_opts=opts.AxisTickOpts(is_show=True),
                axislabel_opts=opts.LabelOpts(formatter="{value} 万")
            )
        )
    )
    
    line1.overlap(line2)
    return line1

# 6. 纯国产电影年度走势
def create_chinese_movie_chart():
    line = (
        Line(init_opts=opts.InitOpts(width="100%", height="400px", theme=ThemeType.LIGHT))
        .add_xaxis(chinese_yearly['上映年份'].tolist())
        .add_yaxis("票房总额万", chinese_yearly['票房总额万'].tolist(), color='green')
        .add_yaxis("影片数量", chinese_yearly['影片数量'].tolist(), color='red', yaxis_index=1)
        .set_global_opts(
            title_opts=opts.TitleOpts(title="纯国产电影年度票房和数量走势"),
            xaxis_opts=opts.AxisOpts(name="上映年份"),
            yaxis_opts=opts.AxisOpts(name="票房总额万"),
            toolbox_opts=opts.ToolboxOpts(is_show=False)
        )
    )
    
    line.extend_axis(
        yaxis=opts.AxisOpts(
            name="影片数量",
            type_="value",
            min_=0,
            max_=chinese_yearly['影片数量'].max() * 1.1,
            position="right",
            axisline_opts=opts.AxisLineOpts(
                linestyle_opts=opts.LineStyleOpts(color="red")
            ),
            axistick_opts=opts.AxisTickOpts(is_show=True),
            axislabel_opts=opts.LabelOpts(formatter="{value} 部")
        )
    )
    return line

# 7. 年度类别票房数据表（使用pyecharts.components.Table）
def create_genre_table_chart():
    # 准备表格数据
    table_data = [["上映年份", "类型", "票房总额万"]]
    for _, row in yearly_genre_box_office.iterrows():
        table_data.append([row['上映年份'], row['主要类型'], f"{row['累计票房万']:.1f}"])
    
    table = Table()
    table.add("年度类别票房", table_data)
    table.set_global_opts(
        title_opts=opts.TitleOpts(title="各个年度各个类别的票房数据"),
        toolbox_opts=opts.ToolboxOpts(is_show=False)
    )
    return table

# 8. 电影类型数量饼图
def create_genre_pie_chart():
    pie = (
        Pie(init_opts=opts.InitOpts(width="100%", height="400px", theme=ThemeType.LIGHT))
        .add(
            "类型分布",
            [list(z) for z in zip(genre_movie_count['类型'], genre_movie_count['影片数量'])],
            radius=["30%", "75%"],
            label_opts=opts.LabelOpts(formatter="{b}: {d}%")
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(title="各个类型电影数量占比"),
            legend_opts=opts.LegendOpts(pos_bottom="1%", pos_left="center"),
            toolbox_opts=opts.ToolboxOpts(is_show=False)
        )
    )
    return pie

# 创建大屏布局
def create_dashboard():
    # 初始化Page
    page = Page(layout=Page.DraggablePageLayout, page_title="中国电影数据可视化大屏")
    
    # 添加各个图表到Page
    page.add(
        create_overview_chart(),
        create_top10_chart(),
        create_country_bar_chart(),
        create_yearly_combined_chart(),
        create_company_chart(),
        create_chinese_movie_chart(),
        create_genre_table_chart(),
        create_genre_pie_chart()
    )
    
    # 保存为HTML文件
    page.render("./movie_data_dashboard.html")
    print("数据可视化大屏已保存为movie_data_dashboard.html")

# 执行函数创建大屏
create_dashboard()

读取数据失败: [Errno 2] No such file or directory: './中国电影网电影_data (1).xlsx'


TypeError: aggregate() missing 1 required positional argument: 'arg'

In [4]:
import pandas as pd
import numpy as np
from pyecharts import options as opts
from pyecharts.charts import Bar, Line, Pie, Grid, Page, Timeline, Scatter, EffectScatter, Map
from pyecharts.globals import ThemeType
from pyecharts.components import Table
import os

# 读取Excel数据
try:
    df = pd.read_excel('./中国电影网电影_data (1).xlsx')
    print(f"成功读取数据，共{len(df)}条记录")
    df.head()
except Exception as e:
    print(f"读取数据失败: {e}")
    # 创建模拟数据用于演示
    df = pd.DataFrame({
        '电影名': [f'电影{i}' for i in range(1, 2625)],
        '累计票房': np.random.uniform(1000, 130000, 2624),
        '导演': ['导演' + str(i % 100) for i in range(2624)],
        '主演': ['主演' + str(i % 200) for i in range(2624)],
        '上映时间': pd.date_range(start='2010-01-01', end='2022-12-31', periods=2624),
        '国家及地区': np.random.choice(['中国', '中国/中国香港', '中国/美国', '美国', '英国'], 2624),
        '发行公司': ['公司' + str(i % 50) for i in range(2624)],
        '类型': np.random.choice(['喜剧', '动作', '爱情', '科幻', '悬疑', '剧情', '动画'], 2624)
    })
    df['累计票房'] = df['累计票房'].round(1)
    df['上映时间'] = df['上映时间'].dt.strftime('%Y-%m-%d')

# 数据预处理
df['累计票房万'] = df['累计票房'].astype(float)
df['上映年份'] = pd.to_datetime(df['上映时间']).dt.year
df['国家'] = df['国家及地区'].str.split('/').str[0]
df['主要类型'] = df['类型'].str.split('/').str[0]

# 1. 业绩总览：累计票房收入（亿元）、各年度占比
total_box_office = df['累计票房万'].sum() / 10000  # 转换为亿元

# 计算各年度票房占比
yearly_box_office = df.groupby('上映年份')['累计票房万'].sum()
total_yearly = yearly_box_office.sum()
yearly_percentage = (yearly_box_office / total_yearly * 100).round(2)

# 2016-2018年度占比
years = [2016, 2017, 2018]
percentage_data = []
for year in years:
    if year in yearly_percentage.index:
        percentage_data.append((f"{year}年度", yearly_percentage[year]))
    else:
        percentage_data.append((f"{year}年度", 0))

# 2. 左一：票房前十影片排行榜
top10_box_office = df.nlargest(10, '累计票房万', keep='all')
top10_box_office = top10_box_office.sort_values('累计票房万', ascending=False)

# 3. 左二：各个国家影片数量的柱状图
country_movie_count = df['国家'].value_counts().reset_index()
country_movie_count.columns = ['国家', '影片数量']

# 4. 中二：各个年度影片票房总额和影片数量（柱状+线性图表）
yearly_stats = df.groupby('上映年份').agg(
    票房总额万=('累计票房万', 'sum'),
    影片数量=('电影名', 'count')
).reset_index()

# 5. 中三：各个电影公司的影片数量和票房总额
company_stats = df.groupby('发行公司').agg(
    影片数量=('电影名', 'nunique'),  # 使用nunique替代count
    票房总额万=('累计票房万', 'sum')
).reset_index()
company_stats = company_stats.nlargest(10, '影片数量', keep='all')

# 6. 右一：纯国产电影各个年度的票房总额和数量走势
chinese_movies = df[df['国家'] == '中国']
chinese_yearly = chinese_movies.groupby('上映年份').agg(
    票房总额万=('累计票房万', 'sum'),
    影片数量=('电影名', 'nunique')  # 使用nunique替代count
).reset_index()

# 7. 右二：各个年度各个影片类别的票房数据表
yearly_genre_box_office = df.groupby(['上映年份', '主要类型'])['累计票房万'].sum().reset_index()

# 8. 右三：各个类别的电影数量饼图
genre_movie_count = df['主要类型'].value_counts().reset_index()
genre_movie_count.columns = ['类型', '影片数量']

# 创建各个图表

# 1. 业绩总览图表
def create_overview_chart():
    pie = (
        Pie(init_opts=opts.InitOpts(width="100%", height="300px", theme=ThemeType.LIGHT))
        .add(
            "票房占比",
            percentage_data,
            radius=["30%", "75%"],
            label_opts=opts.LabelOpts(formatter="{b}: {d}%")
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(
                title="业绩总览",
                subtitle=f"累计票房收入: {total_box_office:.2f} 亿元",
                pos_left="center",
                title_textstyle_opts=opts.TextStyleOpts(font_size=20)
            ),
            legend_opts=opts.LegendOpts(pos_bottom="1%", pos_left="center"),
            toolbox_opts=opts.ToolboxOpts(is_show=False)
        )
    )
    return pie

# 2. 票房前十影片排行榜
def create_top10_chart():
    bar = (
        Bar(init_opts=opts.InitOpts(width="100%", height="400px", theme=ThemeType.LIGHT))
        .add_xaxis(top10_box_office['电影名'].tolist())
        .add_yaxis("累计票房万", top10_box_office['累计票房万'].tolist(), color='skyblue')
        .reversal_axis()
        .set_series_opts(
            label_opts=opts.LabelOpts(position="right")
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(title="票房前十影片排行榜"),
            xaxis_opts=opts.AxisOpts(name="累计票房万"),
            yaxis_opts=opts.AxisOpts(name="电影名"),
            toolbox_opts=opts.ToolboxOpts(is_show=False)
        )
    )
    return bar

# 3. 各个国家影片数量柱状图
def create_country_bar_chart():
    bar = (
        Bar(init_opts=opts.InitOpts(width="100%", height="400px", theme=ThemeType.LIGHT))
        .add_xaxis(country_movie_count['国家'].tolist())
        .add_yaxis("影片数量", country_movie_count['影片数量'].tolist(), color='orange')
        .set_global_opts(
            title_opts=opts.TitleOpts(title="各个国家影片数量对比"),
            xaxis_opts=opts.AxisOpts(name="国家"),
            yaxis_opts=opts.AxisOpts(name="影片数量"),
            toolbox_opts=opts.ToolboxOpts(is_show=False)
        )
    )
    return bar

# 4. 各年度票房总额和数量图表
def create_yearly_combined_chart():
    bar = (
        Bar(init_opts=opts.InitOpts(width="100%", height="400px", theme=ThemeType.LIGHT))
        .add_xaxis(yearly_stats['上映年份'].tolist())
        .add_yaxis("票房总额万", yearly_stats['票房总额万'].tolist(), color='green')
        .set_global_opts(
            title_opts=opts.TitleOpts(title="各年度影片票房总额和数量"),
            xaxis_opts=opts.AxisOpts(name="上映年份"),
            yaxis_opts=opts.AxisOpts(name="票房总额万"),
            toolbox_opts=opts.ToolboxOpts(is_show=False)
        )
    )
    
    line = (
        Line()
        .add_xaxis(yearly_stats['上映年份'].tolist())
        .add_yaxis("影片数量", yearly_stats['影片数量'].tolist(), color='red',
                  yaxis_index=1)
        .set_global_opts(
            yaxis_opts=opts.AxisOpts(
                name="影片数量",
                type_="value",
                min_=0,
                max_=yearly_stats['影片数量'].max() * 1.1,
                position="right",
                axisline_opts=opts.AxisLineOpts(
                    linestyle_opts=opts.LineStyleOpts(color="red")
                ),
                axistick_opts=opts.AxisTickOpts(is_show=True),
                axislabel_opts

SyntaxError: unexpected EOF while parsing (<ipython-input-4-053d28bc9245>, line 177)

In [5]:
import pandas as pd
import numpy as np
from pyecharts import options as opts
from pyecharts.charts import Bar, Line, Pie, Grid, Page, Timeline, Scatter, EffectScatter, Map
from pyecharts.globals import ThemeType
from pyecharts.components import Table
import os

# 读取Excel数据
try:
    df = pd.read_excel('./中国电影网电影_data (1).xlsx')
    print(f"成功读取数据，共{len(df)}条记录")
    df.head()
except Exception as e:
    print(f"读取数据失败: {e}")
    # 创建模拟数据用于演示
    df = pd.DataFrame({
        '电影名': [f'电影{i}' for i in range(1, 2625)],
        '累计票房': np.random.uniform(1000, 130000, 2624),
        '导演': ['导演' + str(i % 100) for i in range(2624)],
        '主演': ['主演' + str(i % 200) for i in range(2624)],
        '上映时间': pd.date_range(start='2010-01-01', end='2022-12-31', periods=2624),
        '国家及地区': np.random.choice(['中国', '中国/中国香港', '中国/美国', '美国', '英国'], 2624),
        '发行公司': ['公司' + str(i % 50) for i in range(2624)],
        '类型': np.random.choice(['喜剧', '动作', '爱情', '科幻', '悬疑', '剧情', '动画'], 2624)
    })
    df['累计票房'] = df['累计票房'].round(1)
    df['上映时间'] = df['上映时间'].dt.strftime('%Y-%m-%d')

# 数据预处理
df['累计票房万'] = df['累计票房'].astype(float)
df['上映年份'] = pd.to_datetime(df['上映时间']).dt.year
df['国家'] = df['国家及地区'].str.split('/').str[0]
df['主要类型'] = df['类型'].str.split('/').str[0]

# 1. 业绩总览：累计票房收入（亿元）、各年度占比
total_box_office = df['累计票房万'].sum() / 10000  # 转换为亿元

# 计算各年度票房占比
yearly_box_office = df.groupby('上映年份')['累计票房万'].sum()
total_yearly = yearly_box_office.sum()
yearly_percentage = (yearly_box_office / total_yearly * 100).round(2)

# 2016-2018年度占比
years = [2016, 2017, 2018]
percentage_data = []
for year in years:
    if year in yearly_percentage.index:
        percentage_data.append((f"{year}年度", yearly_percentage[year]))
    else:
        percentage_data.append((f"{year}年度", 0))

# 2. 左一：票房前十影片排行榜
top10_box_office = df.nlarg

读取数据失败: [Errno 2] No such file or directory: './中国电影网电影_data (1).xlsx'


AttributeError: 'DataFrame' object has no attribute 'nlarg'